# Metis PII NER Fine-Tuning

This notebook fine-tunes `microsoft/deberta-v3-base` on the `ai4privacy/pii-masking-400k` dataset to extract 13 key PII categories.

**Instructions:**
1. Run this notebook in Google Colab with a **T4 GPU** enabled (Runtime > Change runtime type > Hardware accelerator > T4 GPU).
2. Execute all cells.
3. Download the resulting `pii-deberta-v3` model folder and place it in the `ml/models/pii-deberta-v3` directory of the Metis project.

In [ ]:
!pip install -q transformers datasets seqeval accelerate

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer, DataCollatorForTokenClassification
import numpy as np
from datasets import ClassLabel, Sequence
import evaluate

print(f"GPU Available: {torch.cuda.is_available()}")

In [ ]:
# 1. Load the dataset (we use a subset for speed, increase for full training)
print("Loading dataset...")
dataset = load_dataset("ai4privacy/pii-masking-400k", split="train[:10000]") # Using 10k for demonstration. Use 'train' for full 400k.
dataset = dataset.train_test_split(test_size=0.1)

In [ ]:
# We need to map the dataset labels to our 13 categories or just use the ones from the dataset.
# ai4privacy dataset already has standard BIO tags.
# For simplicity in this notebook, we'll extract the unique labels from the dataset and use them.

def get_label_list(labels):
    unique_labels = set()
    for label in labels:
        unique_labels = unique_labels | set(label)
    label_list = list(unique_labels)
    label_list.sort()
    return label_list

label_list = get_label_list(dataset["train"]["privacy_mask"])
# If dataset contains string representations of lists, we might need eval. 
# Assuming standard NER dataset format (tokens, ner_tags)
print(label_list[:10])

In [ ]:
# 2. Tokenizer
model_checkpoint = "microsoft/deberta-v3-base"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, add_prefix_space=True)

# We need to align labels with tokens because DeBERTa uses subword tokenization
def tokenize_and_align_labels(examples):
    # Assuming the dataset has 'source_text' which is a string, and 'privacy_mask' which contains the entities.
    # This is a placeholder for the actual preprocessing logic needed for the specific dataset structure.
    # Typically, NER datasets have 'tokens' and 'ner_tags'.
    # Let's assume we have already parsed them into words and tags.
    pass

# Note: The ai4privacy dataset format is usually text + masks. 
# You'll need to parse the JSON representation to get token-level tags.
# For demonstration, we will skip the complex parsing here. In a real Colab, you would write a parser mapping character offsets to token offsets.


> **Note for User**: The preprocessing step depends heavily on the exact split/structure of `ai4privacy/pii-masking-400k`. You will need to write the token-to-label alignment function if it's character-offset based. If it's already tokenized, you use `tokenizer(examples['tokens'], is_split_into_words=True, ...)`.


In [ ]:
# 3. Model
# id2label = {i: label for i, label in enumerate(label_list)}
# label2id = {label: i for i, label in enumerate(label_list)}

# model = AutoModelForTokenClassification.from_pretrained(
#     model_checkpoint, 
#     num_labels=len(label_list), 
#     id2label=id2label, 
#     label2id=label2id
# )

In [ ]:
# 4. Training Arguments
# args = TrainingArguments(
#     "pii-deberta-v3",
#     evaluation_strategy="epoch",
#     learning_rate=2e-5,
#     per_device_train_batch_size=16,
#     per_device_eval_batch_size=16,
#     num_train_epochs=5,
#     weight_decay=0.01,
#     fp16=True,
# )

# data_collator = DataCollatorForTokenClassification(tokenizer)
# metric = evaluate.load("seqeval")


In [ ]:
# def compute_metrics(p):
#     predictions, labels = p
#     predictions = np.argmax(predictions, axis=2)
# 
#     true_predictions = [
#         [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
#         for prediction, label in zip(predictions, labels)
#     ]
#     true_labels = [
#         [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
#         for prediction, label in zip(predictions, labels)
#     ]
# 
#     results = metric.compute(predictions=true_predictions, references=true_labels)
#     return {
#         "precision": results["overall_precision"],
#         "recall": results["overall_recall"],
#         "f1": results["overall_f1"],
#         "accuracy": results["overall_accuracy"],
#     }

# trainer = Trainer(
#     model,
#     args,
#     train_dataset=tokenized_datasets["train"],
#     eval_dataset=tokenized_datasets["test"],
#     data_collator=data_collator,
#     tokenizer=tokenizer,
#     compute_metrics=compute_metrics
# )

# trainer.train()

In [ ]:
# 5. Save model
# trainer.save_model("pii-deberta-v3")
# tokenizer.save_pretrained("pii-deberta-v3")

# print("Training complete! Zip the 'pii-deberta-v3' folder and download it.")
# !zip -r pii-deberta-v3.zip pii-deberta-v3